<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [ ]:
from ui_lib_strongSort import *

# path of the body detection model
body_model_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/Body_detection_models/Body_detection_model.pt"

# video paths:
# input video directory (without any annotation)
input_video_directory = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input"
# output (final version - with human interaction)
output_video_directory = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Output"

ignore_S1 = [ # related to step 1
    "1",
    "2",
    "3",
    "4",
    "4_cropped",
    "5",
    "6",
    "7",
    "8",
    "12h41_full.MP4",
    "12h41_short.MP4",
    #"20241019 - 13h28.MP4"
]

ignore_S2 = [ # related to step 2
    "1",
    "2",
    "3",
    "4",
    "4_cropped",
    "5",
    "6",
    "7",
    "8",
    "12h41_full.MP4",
    "12h41_short.MP4"
    #"20241019 - 13h28.MP4"
]

ignore_S3 = [ # related to step 3
    "1",
    "2",
    "3",
    "4",
    "4_cropped",
    "5",
    "6",
    "7",
    "8",
    "12h41_full.MP4",
    "12h41_short.MP4",
    #"20241019 - 13h28.MP4"
] 

def has_audio_stream(video_path: str) -> bool:
    """
    Optional: quick check to avoid mux when there is no audio.
    If your mux_audio already tolerates missing audio, you can skip this.
    """
    try:
        import subprocess, json
        probe_cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json",
            video_path,
        ]
        out = subprocess.check_output(probe_cmd).decode("utf-8")
        data = json.loads(out)
        streams = data.get("streams", [])
        return len(streams) > 0
    except Exception:
        # Fallback: assume audio exists to keep behavior; mux_audio should handle errors gracefully
        return True



2026-02-08 20:39:21.538 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:56 | __init__ - BaseTracker initialization parameters:
2026-02-08 20:39:21.539 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:57 | __init__ - det_thresh: 0.3
2026-02-08 20:39:21.539 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:58 | __init__ - max_age: 150
2026-02-08 20:39:21.539 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:59 | __init__ - max_obs: 50
2026-02-08 20:39:21.539 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Ma

<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [ ]:
# FIRST STEP CODE (double-click to extend)

# Directories (same as before)
mannual_annotations_directory = f"{input_video_directory}/manual_annotations"
output_video_directory_temp = f"{output_video_directory}/temp"
raw_text_output_directory = f"{output_video_directory_temp}/raw_output"
treated_directory = f"{output_video_directory}/treated"
final_directory = f"{output_video_directory}/final"


# Create the directories if they do not exist yet
os.makedirs(input_video_directory, exist_ok=True)
os.makedirs(output_video_directory, exist_ok=True)
os.makedirs(mannual_annotations_directory, exist_ok=True)
os.makedirs(output_video_directory_temp, exist_ok=True)
os.makedirs(raw_text_output_directory, exist_ok=True)

# YOLOv8s initialisation (same)
YOLOv8s = YOLO(body_model_path)

# StrongSORT initialisation (no manual OSNet or DeepSORT metric needed)
# If you have a reid weights .pt, point to it; otherwise set to None and StrongSORT may auto-handle/download defaults
reid_weights_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/Re-ID_models/osnet_ain_x1_0_imagenet.pth"  # e.g. "/path/to/osnet_x0_25_msmt17.pt" if you have one
configuration_file_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/config/strongsort_config.yaml"  # e.g. "/path/to/strong_sort.yaml" if you have a custom config; otherwise set to None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
strongsort = build_strongsort(reid_weights=reid_weights_path, device=device, fp16=False, tracker_config_path=configuration_file_path)

for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file_path = f"{mannual_annotations_directory}/{video_name}.txt"
    try:
        with open(annotation_file_path, "x") as f:
            print(f"{video_name}.txt automatically created in {mannual_annotations_directory}.")
    except FileExistsError:
        print(f"{video_name}.txt already present in {mannual_annotations_directory}.")
    print()

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"
    perform_tracking(
        input_video_path=full_video_path,
        output_text_file_path=raw_txt_path,
        detection_model=YOLOv8s,
        tracker=strongsort,
        confidence_threshold=0.5
    )
    print(f"Annotations ready for video: {full_video_path}.\n")

    processed_with_audio = f"{output_video_directory_temp}/{video_name}-(temp)-audio.mp4"

    # Draw and mux (only one output, with audio when available)
    draw_bbox_from_file(
        file_path=raw_txt_path,
        input_video_path=full_video_path,
        output_video_path=processed_with_audio,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, processed_with_audio, processed_with_audio)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

self.max_obs 155
1.mp4 ignored
3.mp4 ignored
2.mp4 ignored
4_cropped.mp4 ignored
4.mp4 ignored
5.mp4 ignored
6.mp4 ignored
7.mp4 ignored
8.mp4 ignored
12h41_short.txt already present in /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/manual_annotations.



Tracking progress (12h41_short.MP4): 100%|█████████▉| 1732/1734 [04:43<00:00,  6.11it/s]


Annotations ready for video: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41_short.MP4.



Drawing annotations (12h41_short.MP4): 100%|█████████▉| 1732/1734 [00:28<00:00, 61.73it/s]


Adding audio...


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -flto=auto -ffat-lto-objects -fexceptions -g -grecord-gcc-switches -pipe -Wall -Werror=format-security -Wno-error=incompatible-pointer-types -Wp,-U_FORTIFY_SOURCE,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -fstack-protector-strong -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -march=x86-64 -mtune=generic -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -mtls-dialect=gnu2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro -Wl,--as-needed -Wl,-z,pack-relative-relocs -Wl,-z,now -specs=/usr/lib/rpm/redhat/redhat-hardened-ld -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -Wl,--build-id

Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41_short.MP4.

12h41_full.txt already present in /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/manual_annotations.



Tracking progress (12h41_full.MP4):  10%|█         | 1093/10536 [03:15<09:15, 17.00it/s]

<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [ ]:
# ---------- STEP 2: apply manual edits -> treated output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-treated.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

1.mp4 ignored
3.mp4 ignored
2.mp4 ignored
4_cropped.mp4 ignored
4.mp4 ignored
5.mp4 ignored
6.mp4 ignored
7.mp4 ignored
8.mp4 ignored


Drawing annotations (12h41.MP4): 100%|█████████▉| 1732/1734 [00:20<00:00, 86.06it/s]

Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41.MP4.



### Third step

In [ ]:
# ---------- STEP 3: final arrows/names -> final output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-final.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="triangle",
        draw_frame_count=False,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")